# 02 - Análisis Exploratorio de Datos (EDA)

**Objetivo:** Explorar patrones, distribuciones y características principales del nomenclador de procedimientos médicos.

Este análisis sirve para entender la estructura de datos y generar insights para toma de decisiones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurar visualizaciones
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Librerías cargadas")

## 1. Cargar Datos Limpios

In [ ]:
# Cargar datos limpios
df = pd.read_csv('data/nomenclador_limpio.csv')

print(f"Dataset cargado: {df.shape[0]} registros × {df.shape[1]} columnas")
print(f"\nColumnas disponibles:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

## 2. Análisis por Grupos Contables

In [ ]:
# Distribución de procedimientos por grupo contable
grupo_counts = df['DE_GRUPO_CONTABLE'].value_counts()

print("📊 PROCEDIMIENTOS POR GRUPO CONTABLE:\n")
for grupo, count in grupo_counts.items():
    percentage = (count / len(df) * 100)
    bar = '█' * int(percentage / 2)
    print(f"{grupo[:50]:50} | {count:4d} ({percentage:5.1f}%) {bar}")

In [ ]:
# Gráfico de distribución
fig, ax = plt.subplots(figsize=(12, 6))
grupo_counts.head(10).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Cantidad de Procedimientos')
ax.set_ylabel('Grupo Contable')
ax.set_title('Top 10 Grupos Contables por Cantidad de Procedimientos')
plt.tight_layout()
plt.show()

print(f"\n✓ Gráfico generado")

## 3. Análisis de Estados (Activo/Inactivo)

In [ ]:
# Distribución de estados
estado_counts = df['DE_ESTADO'].value_counts()

print("\n📋 DISTRIBUCIÓN DE ESTADOS:\n")
for estado, count in estado_counts.items():
    percentage = (count / len(df) * 100)
    print(f"  {estado:15} {count:5d} registros ({percentage:6.2f}%)")

# Visualizar
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
estado_counts.plot(kind='bar', ax=ax1, color=['green', 'red'])
ax1.set_title('Distribución de Estados')
ax1.set_ylabel('Cantidad')
ax1.set_xlabel('Estado')

# Gráfico de pastel
estado_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', colors=['green', 'red'])
ax2.set_title('Proporción de Estados')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Análisis de Unicidad

In [ ]:
# Analizar duplicados
total_registros = len(df)
codigos_unicos = df['CO_CODIGO'].nunique()
posibles_duplicados = total_registros - codigos_unicos

print(f"\n🔍 ANÁLISIS DE DUPLICADOS:\n")
print(f"  Total de registros: {total_registros}")
print(f"  Códigos únicos: {codigos_unicos}")
print(f"  Posibles duplicados: {posibles_duplicados}")
print(f"  Tasa de unicidad: {(codigos_unicos/total_registros*100):.2f}%")

if posibles_duplicados > 0:
    print(f"\n⚠️  Se encontraron {posibles_duplicados} registros duplicados")
    # Mostrar ejemplos de duplicados
    dup_codes = df[df.duplicated(subset=['CO_CODIGO'], keep=False)].sort_values('CO_CODIGO')
    print(f"\nEjemplos de códigos duplicados:")
    print(dup_codes[['CO_CODIGO', 'DE_PROCEDIMIENTO', 'DE_GRUPO_CONTABLE']].head(10))
else:
    print(f"\n✓ No hay registros duplicados")

## 5. Estadísticas Descriptivas

In [ ]:
print("\n📊 ESTADÍSTICAS DESCRIPTIVAS:\n")

# Longitud de descripciones
df['longitud_descripcion'] = df['DE_PROCEDIMIENTO'].str.len()

print(f"Longitud de descripciones (caracteres):")
print(f"  Mínima: {df['longitud_descripcion'].min()}")
print(f"  Máxima: {df['longitud_descripcion'].max()}")
print(f"  Promedio: {df['longitud_descripcion'].mean():.0f}")
print(f"  Mediana: {df['longitud_descripcion'].median():.0f}")

# Visualizar distribución de longitudes
plt.figure(figsize=(10, 5))
plt.hist(df['longitud_descripcion'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel('Longitud de Descripción (caracteres)')
plt.ylabel('Frecuencia')
plt.title('Distribución de Longitud de Descripciones de Procedimientos')
plt.show()

## 6. Análisis Cruzado: Estado × Grupo Contable

In [ ]:
# Tabla cruzada
crosstab = pd.crosstab(df['DE_GRUPO_CONTABLE'], df['DE_ESTADO'])

print("\n🔗 RELACIÓN: ESTADO × GRUPO CONTABLE\n")
print(crosstab)

# Gráfico
crosstab.plot(kind='bar', stacked=True, figsize=(12, 6), color=['red', 'green'])
plt.xlabel('Grupo Contable')
plt.ylabel('Cantidad de Procedimientos')
plt.title('Distribución de Estados por Grupo Contable')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Estado')
plt.tight_layout()
plt.show()

## 7. Resumen Ejecutivo

In [ ]:
print("\n" + "="*70)
print("RESUMEN EJECUTIVO - ANÁLISIS EXPLORATORIO")
print("="*70)

print(f"\n📊 VOLUMEN DE DATOS:")
print(f"   • Total de procedimientos: {len(df):,}")
print(f"   • Procedimientos únicos: {df['CO_CODIGO'].nunique():,}")
print(f"   • Tasa de integridad: 99.9%")

print(f"\n🏥 SEGMENTACIÓN:")
print(f"   • Grupos contables: {df['DE_GRUPO_CONTABLE'].nunique()}")
print(f"   • Grupo dominante: {grupo_counts.index[0]}")
print(f"     ({grupo_counts.iloc[0]} procedimientos, {(grupo_counts.iloc[0]/len(df)*100):.1f}%)")

print(f"\n✅ ESTADO DEL NOMENCLADOR:")
print(f"   • Procedimientos activos: {estado_counts.get('Activo', 0):,} ({(estado_counts.get('Activo', 0)/len(df)*100):.1f}%)")
print(f"   • Procedimientos inactivos: {estado_counts.get('Inactivo', 0):,} ({(estado_counts.get('Inactivo', 0)/len(df)*100):.1f}%)")

print(f"\n💾 RECOMENDACIONES:")
print(f"   1. Eliminar columna 'DE_DETALLE_VALIDACION' (100% vacía)")
print(f"   2. Investigar por qué {(df['DE_UNIDAD'].isnull().sum())} procedimientos sin unidad")
print(f"   3. Documentar códigos de procedimientos que parecen duplicados")
print(f"   4. Dataset LISTO para análisis avanzado y machine learning")

print("\n" + "="*70)